# Map-Reduce Summarization and Confidence Signaling

Synthesizing thousands of transcripts into insights people can trust.

This notebook is an original tutorial extending the course with current
(2026) state-of-the-art practice, interview-focused.

## Learning Objectives

- Explain the map-reduce summarization pattern and when it beats long-context stuffing.
- Know why lost-in-the-middle still matters on 1M-token models.
- Design confidence signaling: support counts, evidence spans, small-n guardrails, confidence tiers.
- Explain why verbalized model confidence is unreliable and what to use instead.

## Mental Model

Long context made chunking optional; it did **not** make map-reduce obsolete.
Map-reduce survives because it buys four things a giant single call cannot:

1. **Parallelism** — map thousands of interviews concurrently; minutes, not hours.
2. **Cost shaping** — map with a cheap model, reduce with a frontier model; cache per-interview summaries and reuse them across every dashboard query.
3. **Citation traceability** — each map output carries interview ids and quote spans, so the reduce step can only cite what the maps produced. A 2,000-transcript mega-prompt makes attribution unverifiable.
4. **Uniform attention** — every interview gets the same processing budget instead of positional luck (lost-in-the-middle).

Confidence, meanwhile, is not a number the model tells you — it is a
*structure you build*: exact support counts from classification, verified
quotes, and honest small-n labeling.

## Important Concepts

- **Map-reduce**: per-unit structured extraction (map) → group → per-theme synthesis (reduce) → top-level report. Two reduce levels max; more loses fidelity.
- **Refine/iterative**: fold documents into a running summary sequentially — order-sensitive, non-parallel, mostly obsolete except genuinely sequential material (longitudinal diaries).
- **Lost-in-the-middle**: recall degrades with input length on all long-context models (~60% average recall at 1M tokens in RULER-style evals).
- **Prevalence vs intensity**: how many said it vs how strongly they said it — report both, conflate neither.
- **Verbalized confidence**: RLHF'd models say "90% confident" with calibration error up to 0.30+; unreliable alone.
- **Self-consistency**: sample the classification k times; agreement rate is a usable confidence signal.
- **Quote verification**: programmatically check cited quotes exist in the source transcript (exact/fuzzy match) — cheap, catches hallucinated evidence.

## Need-To-Know Coverage Checklist

- [x] Chunk → summarize → summarize-of-summaries, and the modern per-interview → per-theme → report shape.
- [x] When map-reduce wins vs long-context stuffing (parallelism, cost, citations, attention).
- [x] Refine pattern and why long context killed it more than map-reduce.
- [x] Support quantification as the primary confidence signal ("23 of 412 respondents").
- [x] Ensemble confidence: self-consistency + logprobs + verbalized as tiebreaker.
- [x] Small-n guardrails: thresholds, hedged phrasing, denominators everywhere.
- [x] Dashboard patterns: confidence tiers, sample-size warnings on filters, click-through to quotes.

## Deep Study Notes

### The canonical shape for an interview product

```
per-interview MAP (cheap model):
    structured extraction — summary + tagged quotes (with timestamps) + facets
        │  cached; reused by every downstream query
        ▼
group by theme / question
        ▼
per-theme REDUCE (frontier model):
    synthesis over the extracts, citation-constrained to mapped quotes
        ▼
top-level report over theme syntheses
```

A 45-minute interview is ~10k tokens — no chunking needed *within* an
interview anymore. The map unit is the interview, not the chunk.

### Why not one giant call?

- Lost-in-the-middle persists in 2026: all tested models degrade with input
  length; stuffing 2,000 transcripts into 1M tokens is unreliable even where
  it fits.
- Attribution: you cannot verify which transcript a claim came from.
- Cost: re-running the mega-call for every dashboard question vs reusing
  cached per-interview extracts.
- Wall-clock: maps parallelize; one giant call does not.

### Confidence and uncertainty signaling

- **Support counts are the primary signal.** Because classification assigns
  every response to themes, each insight has an exact count: "23 of 412
  respondents (5.6%)". Show prevalence bars, not just theme names.
- **Prevalence vs intensity — measure and report both, conflate neither.**
  *Prevalence* is how many respondents expressed the theme (the count above).
  *Intensity* is how strongly each one expressed it — measured by adding a
  per-mention field to the map schema (e.g. `intensity: mild | moderate |
  severe`, or sentiment strength, classified by the same cheap model that
  assigns the theme) and aggregating it separately. The two diverge in
  decision-relevant ways: a theme mentioned by 40% mildly ("pricing is a bit
  confusing") is a different problem from one mentioned by 8% severely ("I
  churned over the billing surprise"). A dashboard that shows only prevalence
  buries the second; report count *and* an intensity distribution per theme.
- **Verbalized confidence is weak alone**: models emit 80-100% confidence with
  large calibration error. Never ship raw "the model is 85% confident".
  Use an ensemble: self-consistency agreement rate (primary), token logprobs
  where available, verbalized as tiebreaker.
- **Evidence spans**: every insight cites verbatim quotes with interview ids
  and audio timestamps (the "highlight reel" pattern). **Verify quotes
  programmatically** — exact/fuzzy match against the transcript catches
  hallucinated evidence cheaply.
- **Small-n guardrails**: insights below a support threshold (n<5 or <3%)
  render as "emerging signal" badges, greyed or sectioned separately. Forbid
  universal quantifiers ("users want...") in synthesis prompts; require
  count-tied hedged phrasing. Show denominators everywhere (n asked vs n
  answered vs n coded). Slicing a dashboard filter down to n=4 must visibly warn.
- **Provenance**: log model, prompt/schema version, temperature, and run
  timestamp per extraction; inter-run classification agreement doubles as a
  calibration proxy.

## Common Failure Modes

- Three or more reduce levels → each level compresses away nuance; by level 4 everything is generic.
- Reduce step allowed to introduce claims not present in any map output → unverifiable synthesis.
- "Users want X" from n=3 → over-generalization; the single most damaging dashboard failure.
- Raw verbalized confidence on the dashboard → false precision users learn to distrust.
- Quotes not verified against transcripts → hallucinated evidence discovered by a customer.
- No caching of map outputs → every dashboard question re-processes the corpus.

## Implementation Notes

- Map outputs are structured (schema from the previous notebook) and cached keyed by (interview_id, prompt_version).
- The reduce prompt receives *only* map outputs, never raw transcripts — this enforces citation discipline structurally.
- Store confidence tier (established / supported / emerging) as data, not as prose in the summary.

In [ ]:
"""Map-reduce synthesis with support counts, quote verification, and tiers."""
import json

TRANSCRIPTS = {
    "i1": "Honestly the signup flow kept erroring out. I nearly gave up.",
    "i2": "Creating my account took three tries. Once in, it was fine.",
    "i3": "Pricing tiers are confusing, I do not know what I am paying for.",
    "i4": "The signup page crashed twice on my phone.",
}

# --- MAP: per-interview structured extraction (cheap model in prod) -------
MAP_OUTPUTS = [
    {"id": "i1", "theme": "onboarding friction", "quote": "the signup flow kept erroring out"},
    {"id": "i2", "theme": "onboarding friction", "quote": "took three tries"},
    {"id": "i3", "theme": "pricing confusion", "quote": "Pricing tiers are confusing"},
    {"id": "i4", "theme": "onboarding friction", "quote": "signup page crashed twice"},
    # A hallucinated quote, to show verification catching it:
    {"id": "i3", "theme": "pricing confusion", "quote": "I would pay double for clarity"},
]


def verify_quote(interview_id: str, quote: str) -> bool:
    """Cheap anti-hallucination check: the quote must exist in the source."""
    return quote.lower() in TRANSCRIPTS.get(interview_id, "").lower()


verified = [m for m in MAP_OUTPUTS if verify_quote(m["id"], m["quote"])]
rejected = [m for m in MAP_OUTPUTS if m not in verified]
print(f"verified {len(verified)}/{len(MAP_OUTPUTS)} quotes; rejected: "
      f"{[m['quote'] for m in rejected]}")

# --- REDUCE: per-theme synthesis, citation-constrained to map outputs -----
N_TOTAL = len(TRANSCRIPTS)
themes = {}
for m in verified:
    themes.setdefault(m["theme"], []).append(m)


def tier(n: int, total: int) -> str:
    share = n / total
    if n >= 3 and share >= 0.5:
        return "established"
    if n >= 2:
        return "supported"
    return "emerging signal (low n)"


insights = [
    {
        "theme": theme,
        "support": f"{len(set(m['id'] for m in members))} of {N_TOTAL} respondents",
        "tier": tier(len(set(m["id"] for m in members)), N_TOTAL),
        "evidence": [{"id": m["id"], "quote": m["quote"]} for m in members],
    }
    for theme, members in themes.items()
]
print(json.dumps(insights, indent=2))


## Practice

1. Add fuzzy matching (e.g. normalized token overlap) to `verify_quote` so
   lightly-paraphrased quotes pass but invented ones fail. Where do you set
   the threshold and what error do you prefer (false reject vs false accept)?
2. Add a second reduce level: synthesize a one-paragraph report from the
   insights list — constrained to only mention themes with tier != "emerging".
3. Implement self-consistency: classify one response 5 times with a simulated
   noisy classifier and use agreement rate as the confidence score.
4. A PM asks "why not just paste all 2,000 transcripts into the 1M-context
   model?" Give the four-part answer from the mental model.

## Design Checklist

- [ ] Map outputs are structured, cached, and carry ids + quote spans.
- [ ] Reduce sees only map outputs; citations verified programmatically.
- [ ] At most two reduce levels.
- [ ] Every insight shows support count and denominator.
- [ ] Confidence tiers are data; small-n insights visually distinct.
- [ ] No universal quantifiers in synthesis prompts.
- [ ] Provenance (model, prompt version, timestamp) logged per extraction.